In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# 1. Fetch Data from NYC Open Data (No API Key needed for small limits)
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json?$limit=10000"
response = requests.get(url)
data = response.json()

# 2. Convert to Spark DataFrame
# We use pandas as an intermediary for easy JSON parsing
pdf = pd.DataFrame(data)
raw_df = spark.createDataFrame(pdf)

# 3. Add ingestion metadata
bronze_df = raw_df.withColumn("ingestion_timestamp", current_timestamp())

# 4. Write to Bronze Delta Table (Append Mode)
# Using Unity Catalog managed table (serverless-compatible)
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze_nyc_311")

print(f"Bronze layer loaded successfully to table bronze_nyc_311")

Bronze layer loaded successfully to table bronze_nyc_311


In [0]:
from pyspark.sql.functions import col, to_date, datediff

# 1. Read from Bronze
bronze_df = spark.table("bronze_nyc_311")

# 2. Transform Data
silver_df = bronze_df \
    .withColumn("created_date", to_date(col("created_date"))) \
    .withColumn("closed_date", to_date(col("closed_date"))) \
    .withColumn("resolution_days", datediff(col("closed_date"), col("created_date"))) \
    .filter(col("incident_zip").isNotNull()) \
    .select(
        "unique_key",
        "created_date",
        "closed_date",
        "agency",
        "complaint_type",
        "descriptor",
        "incident_zip",
        "borough",
        "resolution_days"
    )

# 3. Write to Silver Delta Table (Overwrite for this example, or Merge for prod)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_nyc_311")

# 4. Physical Optimization: Z-Ordering
# This colomcates similar zip codes to drastically reduce I/O during aggregations
spark.sql("OPTIMIZE silver_nyc_311 ZORDER BY (incident_zip)")

print("Silver layer cleaned and Z-Ordered.")

Silver layer cleaned and Z-Ordered.


In [0]:
from pyspark.sql.functions import avg, count, round, col

# 1. Read from Silver (managed table from previous step)
silver_df = spark.table("silver_nyc_311")

# 2. Aggregate Data
gold_df = silver_df \
    .filter(col("resolution_days").isNotNull()) \
    .groupBy("borough", "agency", "complaint_type") \
    .agg(
        count("unique_key").alias("total_complaints"),
        round(avg("resolution_days"), 2).alias("avg_resolution_days")
    ) \
    .orderBy("borough", "total_complaints", ascending=[True, False])

# 3. Write to Gold Delta Table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_nyc_311_metrics")

# 4. Display the results
display(spark.table("gold_nyc_311_metrics"))

borough,agency,complaint_type,total_complaints,avg_resolution_days
BRONX,NYPD,Noise - Residential,404,0.08
BRONX,NYPD,Illegal Parking,293,0.12
BRONX,NYPD,Noise - Street/Sidewalk,144,0.11
BRONX,NYPD,Blocked Driveway,111,0.07
BRONX,NYPD,Noise - Commercial,79,0.06
BRONX,NYPD,Abandoned Vehicle,37,0.03
BRONX,NYPD,Noise - Vehicle,30,0.03
BRONX,DSNY,Derelict Vehicles,16,0.13
BRONX,NYPD,Non-Emergency Police Matter,13,0.08
BRONX,DSNY,Vendor Enforcement,12,0.0
